In [71]:
import pandas as pd
import lightgbm as lgb
from sklearn.utils import shuffle

In [72]:
data = pd.read_csv('training_set_VU_DM.csv')

In [73]:
data.fillna(0, inplace=True)
data.sort_values("srch_id", inplace=True)

In [74]:
unique_groups = data['srch_id'].unique()
shuffled_groups = shuffle(unique_groups, random_state=42)

In [75]:
split_index = int(0.8 * len(shuffled_groups))

In [76]:
train_groups = shuffled_groups[:split_index]
test_groups = shuffled_groups[split_index:]

In [77]:
train_df = data[data['srch_id'].isin(train_groups)]
test_df = data[data['srch_id'].isin(test_groups)]

In [78]:
X_train = train_df.drop(['click_bool', 'booking_bool', 'gross_bookings_usd', 'srch_id', 'position', 'date_time'], axis=1)
y_train = train_df['booking_bool']
X_test = test_df.drop(['click_bool', 'booking_bool', 'gross_bookings_usd', 'srch_id', 'position', 'date_time'], axis=1)
y_test = test_df['booking_bool']

In [79]:
groups_train = train_df.groupby('srch_id').size().values
groups_test = test_df.groupby('srch_id').size().values

In [80]:
train_data = lgb.Dataset(X_train, label=y_train, group=groups_train)
test_data = lgb.Dataset(X_test, label=y_test, group=groups_test)

In [81]:
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'verbose': -1,
    'ndcg_eval_at': [1, 3, 5, 10]
}

In [82]:
num_round = 100
bst = lgb.train(params, train_data, num_round, valid_sets=[test_data], callbacks=[lgb.early_stopping(stopping_rounds=10)])

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[61]	valid_0's ndcg@1: 0.45025	valid_0's ndcg@3: 0.539336	valid_0's ndcg@5: 0.580162	valid_0's ndcg@10: 0.623537


In [87]:
test_data = pd.read_csv('test_set_VU_DM.csv')

In [88]:
test_data.fillna(0, inplace=True)
mod_test_data = test_data.drop(['srch_id', 'date_time'], axis=1)

In [89]:
mod_test_data.head()

,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score1,...,comp5_rate_percent_diff,comp6_rate,comp6_inv,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff
0,24,216,0.0,0.0,219,3180,3,4.5,1,2.94,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,24,216,0.0,0.0,219,5543,3,4.5,1,2.64,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,24,216,0.0,0.0,219,14142,2,3.5,1,2.71,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,24,216,0.0,0.0,219,22393,3,4.5,1,2.40,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,24,216,0.0,0.0,219,24194,3,4.5,1,2.94,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [90]:
test_pred = bst.predict(mod_test_data)

In [91]:
test_data['pred'] = test_pred

In [92]:
sorted_results = test_data.sort_values(by=['srch_id', 'pred'], ascending=[True, False])

In [93]:
submit_data = sorted_results[['srch_id', 'prop_id']]

In [94]:
submit_data.to_csv('submission1.csv', index=False)

In [95]:
len(submit_data)

4959183

In [96]:
len(test_data)

4959183